In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import seaborn as sns

# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "src"))
sys.path.insert(0, src_dir)

#import the module
import corr_utils
import warnings
warnings.filterwarnings("ignore")

from joblib import dump, load
from scipy.cluster.hierarchy import fcluster
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram
from sklearn.utils import resample
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.spatial.distance import pdist, squareform
import community as community_louvain

from sklearn.decomposition import PCA

import networkx as nx

In [ ]:
#pip install fastcluster

In [ ]:
# Versions for imported libs
print(f"{'python':<14} {sys.version.split()[0]}")

mods = {
    "pandas": pd,
    "numpy": np,
    "seaborn": sns,
    "matplotlib": getattr(plt, "matplotlib", None) or sys.modules.get("matplotlib"),
    "scikit-learn": sys.modules.get("sklearn"),   # loaded via sklearn.decomposition import
    "scipy": sys.modules.get("scipy"),            # loaded via scipy.cluster/... imports
    "joblib": sys.modules.get("joblib"),
    "networkx": sys.modules.get("networkx"),
    "corr_utils": sys.modules.get("corr_utils") if "corr_utils" in sys.modules else corr_utils,
}

for name, mod in mods.items():
    print(f"{name:<14} {getattr(mod, '__version__', '(no __version__)') if mod else 'not loaded'}")


## Pearson


### Datasets

In [ ]:
# Peason correlation between each protein (rows) with all genes (columns)
# computed usnf 100 bootstrap iterations
pearson = load('data/full_pearson100.joblib')
pearson


### Check +0.5, 0.6, 0.7, 0.8, 0.9 correlation

In [ ]:
import os

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
datasets = {}

output_dir = "pearson proteins"
os.makedirs(output_dir, exist_ok=True)

for t in thresholds:
    df_t = pearson.where(pearson > t)          # keep only values > t
    df_t = df_t.dropna(how='all', axis=0)      # remove rows with no values > t
    df_t = df_t.dropna(how='all', axis=1)      # remove empty columns
    datasets[f'gt_{str(t).replace(".", "")}'] = df_t

    # names present in both rows and columns
    same_names = df_t.index.intersection(df_t.columns)

    print(
        f">{t}: {df_t.shape[0]} rows, {df_t.shape[1]} columns, "
        f"{len(same_names)} row names also present as column names"
    )

    # save only overlapping names, but skip threshold 0.9
    if t != 0.9:
        filename = os.path.join(output_dir, f"protein_list_0{str(t).split('.')[1]}.txt")
        same_names.to_series().to_csv(filename, index=False, header=False)

In [ ]:
same_names.index

In [ ]:
# Filtered transcriptomics data used for modeling to consider only filtered genes
transcriptomics_filtered = load('data/transcriptomics_filtered.joblib')
transcriptomics_filtered

In [ ]:
# Remove genes filtered out in the filtering procedure
pearson = pearson[transcriptomics_filtered.columns]
pearson

In [ ]:
# Compute how many zeros and how many corr abpove thresholds are there
zero_counts = (pearson == 0).sum(axis=1)

# Compute the number of genes correlated above thresholds
thresholds = [0.1, 0.2, 0.3]
thresh_counts = {t: (pearson.abs() > t).sum(axis=1) for t in thresholds}

In [ ]:

# Clean the Pearson correlation matrix

#1. Replace infinite values with NaN.
pearson_clean = pearson.replace([np.inf, -np.inf], np.nan)

# 2. Drop rows and columns with NaN values.
pearson_clean = pearson_clean.dropna(axis=0, how='any').dropna(axis=1, how='any')

# 3. Drop rows and columns with zero variance to avoid division-by-zero issues.
pearson_clean = pearson_clean.loc[pearson_clean.var(axis=1) != 0, :]
pearson_clean = pearson_clean.loc[:, pearson_clean.var(axis=0) != 0]

# Print shapes to ensure cleaning worked.
print("Shape after cleaning:", pearson_clean.shape)


In [ ]:
# Statistical prints
print("Min value:", pearson_clean.min().min())
print("Max value:", pearson_clean.max().max())
# Check distribution of values
print("Negative values percentage:", 
      (pearson_clean < 0).sum().sum() / pearson_clean.size * 100, "%")
print("Infinite values percentage:", 
      np.isinf(pearson_clean).sum().sum() / pearson_clean.size * 100, "%")
print("NaN values percentage:", 
      pearson_clean.isna().sum().sum() / pearson_clean.size * 100, "%")

### Pearson Correlations full distribution 

In [ ]:

A = pearson_clean.to_numpy(dtype=np.float32, copy=False)
iu = np.triu_indices_from(A, k=1)
data = A[iu]
data = data[np.isfinite(data)]
non_zero = data[np.abs(data) > 1e-10]

bins = np.linspace(-1, 1, 51)
h_all, edges = np.histogram(data, bins=bins)
h_nz,  _     = np.histogram(non_zero, bins=bins)
widths = np.diff(edges)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=False)
axes[0].bar(edges[:-1], h_all, width=widths, align="edge"); axes[0].set_title("Unique Pairs")
axes[1].bar(edges[:-1], h_all, width=widths, align="edge"); axes[1].set_title("Unique Pairs (Truncated)"); axes[1].set_ylim(top=50000)
axes[2].bar(edges[:-1], h_nz,  width=widths, align="edge"); axes[2].set_title("Non-zero Unique Pairs")
for ax in axes:
    ax.set_xlabel("Correlation Coefficient"); ax.set_ylabel("Frequency")
plt.tight_layout(); plt.show()


In [ ]:
# Count how many zeros and how many corr above thresholds are there
zero_counts = (pearson_clean == 0).sum(axis=1)

# Compute the number of genes correlated above thresholds
thresholds = [0.1, 0.2, 0.3]
thresh_counts = {t: (pearson_clean.abs() > t).sum(axis=1) for t in thresholds}

In [ ]:
# Helpter function to print summary table
def print_summary():
    summary = pd.DataFrame({"Zero Correlations": zero_counts})
    
    # Compute median number of zero correlations per protein
    median_zeros = zero_counts.median()
    
    for t in thresholds:
        # Count how many correlations are above the threshold per protein
        above_threshold = (pearson_clean.T > t).sum(axis=1)
        
        # Add count and median columns to the summary
        summary[f"Correlated > {t}"] = thresh_counts[t]
        summary[f"Median > {t}"] = above_threshold.median()

    # Print the table
    print(summary)

    # Print medians
    print("\nMedian Number of Zero Correlations per Protein:", median_zeros)
    for t in thresholds:
        median_corr = (pearson.T > t).sum(axis=1).median()
        print(f"Median Number of Correlations > {t} per Protein: {median_corr}")


In [ ]:
print_summary()

### Threshold counts

The following histograms display how many genes per protein have correlations 0.0 and above 0.1, 0.2, and 0.3 (absolute value). It helps to understand whether most genes have weak or strong correlations with each protein.

In [ ]:
# Thresholded histograms
fig, axes = plt.subplots(2, 2, figsize=(10, 6))

# 1) Zero counts
sns.histplot(zero_counts, bins=50, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Distribution of Zero Correlations per Protein")
axes[0, 0].set_xlabel("Number of Zero Correlations")
axes[0, 0].set_ylabel("Count of Proteins")

# 2–4) Thresholded counts
thresholds = [0.1, 0.2, 0.3]
targets = [axes[0, 1], axes[1, 0], axes[1, 1]]

for ax, thr in zip(targets, thresholds):
    sns.histplot(thresh_counts[thr], bins=50, kde=True, ax=ax)
    ax.set_title(f"Proteins with Genes Correlated Above {thr}")
    ax.set_xlabel(f"Count of Correlated Genes (>|{thr}|)")
    ax.set_ylabel("Number of Proteins")

plt.tight_layout()
plt.show()


## Generate Clusters

Row linkage hierarchical clustering builds a dendrogram of the matrix’s rows by iteratively merging the most similar rows/clusters based on a chosen distance metric.
With Ward linkage + Euclidean distance, each merge minimizes the increase in within-cluster variance, yielding compact, roughly spherical clusters.
The resulting linkage matrix records the merge history; cut it at a chosen height to obtain row cluster labels and reorder heatmaps for visualization.

Important distinction between the following 2 cells: the first approach might include effects of any transformations applied by seaborn (like the standard_scale=1 parameter), while the second approach works on data exactly as provided.

In [ ]:

#takes ~ 90 mins to run 

# os.makedirs('results/clustermaps', exist_ok=True)

# # Create the clustermap.
# sns.set_theme(style="white")
# g = sns.clustermap(
#     pearson_clean,
#     method="ward",         # Clustering method
#     metric="euclidean",    # Distance metric
#     cmap="coolwarm",           # Color map for correlation data
#     center=0,              # Center the colormap at zero
#     vmin=-1, vmax=1,       # Expected correlation range
#     #standard_scale=1,      # Standardize each row (protein)
#     figsize=(10, 10)       # Figure size
# )

# # 5. Rotate x-axis labels for readability.
# plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")

# g.savefig('results/clustermaps/map_pg_raw.png', dpi=300, bbox_inches='tight')

# plt.show()


In [ ]:
sns.set_theme(style="white")
g = sns.clustermap(
    pearson_clean,
    method="ward",         # Clustering method
    metric="euclidean",    # Distance metric
    cmap="coolwarm",           # Color map for correlation data
    center=0,              # Center the colormap at zero
    vmin=-1, vmax=1,       # Expected correlation range
    standard_scale=1,      # Standardize each row (protein)
    figsize=(10, 10)       # Figure size
)

# 5. Rotate x-axis labels for readability.
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")

os.makedirs('results/clustermaps', exist_ok=True)
g.savefig('results/clustermaps/map_pg_std.png', dpi=300, bbox_inches='tight')

plt.show()

Dendogram for clustering

Another important distinction: the first approach might include effects of any transformations applied by seaborn (like the standard_scale=1 parameter), while the second approach works on data exactly as provided.

In [ ]:
# Generate the hierarchical linkage matrix from the cleaned Pearson matrix
#gb row_linkage = g.dendrogram_row.linkage
row_linkage = sch.linkage(pearson_clean, method="ward", metric="euclidean")

# 1. Dendrogram Method
corr_utils.plot_dendrogram(row_linkage)  # Visually inspect and manually choose a number

## Evaluate clusters

In [ ]:
#Elbow Method
elbow_k = corr_utils.elbow_method(pearson_clean, row_linkage)

# Silhouette Score Method
silhouette_k = corr_utils.silhouette_method(pearson_clean, row_linkage)

### Consensus Clustering

In [ ]:
# Parameters
num_bootstraps = 3  # Number of bootstrap samples
max_clusters = 7   # Max number of clusters to evaluate

# Number of proteins (rows) in your dataset
n_proteins = pearson_clean.shape[0]  # should be 5113

# Create consensus matrices - correct initialization
consensus_matrices = np.zeros((max_clusters - 1, n_proteins, n_proteins))

# Perform consensus clustering for k = 2 to max_clusters
for k in range(2, max_clusters + 1):
    coassociation_matrix = np.zeros((n_proteins, n_proteins))
    
    # Bootstrap sampling and clustering
    for _ in range(num_bootstraps):
        # Resample proteins (rows) only
        sampled_data = resample(pearson_clean, replace=True)
        row_linkage = sch.linkage(sampled_data, method="ward", metric="euclidean")
        clusters = sch.fcluster(row_linkage, k, criterion="maxclust")
    
        # Update coassociation matrix
        for i in range(len(clusters)):
            for j in range(len(clusters)):
                if clusters[i] == clusters[j]:
                    coassociation_matrix[i, j] += 1
    
    # Normalize to obtain consensus matrix for current k (stored at index k-2)
    consensus_matrices[k - 2] = coassociation_matrix / num_bootstraps

In [ ]:
# Example usage with consensus matrices:
corr_utils.visualize_all_consensus_matrices(consensus_matrices)

### Saving clusters

In [ ]:
# The row dendrogram linkage is stored in g.dendrogram_row.linkage
row_linkage = g.dendrogram_row.linkage

In [ ]:
# ------------------------------
# Create the directory if it doesn't exist
os.makedirs("results/clusters_pg", exist_ok=True)

# Save cluster assignments for all values from 2 to elbow_k
for k in range(2, 5):
    # Get cluster labels for this k
    cluster_labels = fcluster(row_linkage, k, criterion='maxclust')
    
    # Create a DataFrame mapping each protein to its cluster
    clusters_df = pd.DataFrame({
        'Protein': pearson_clean.index,
        'Cluster': cluster_labels
    })
    
    # Define the output file path
    output_file = f"results/clusters_pg/protein_clusters_k{k}.csv"
    
    # Save the cluster assignments to a CSV file
    clusters_df.to_csv(output_file, index=False)
    
    print(f"Saved cluster assignments for k={k} to '{output_file}'")

print(f"\nAll cluster assignments from k=2 to k={elbow_k} have been saved to the 'results/clusters_pg' folder.")


#### number of clusters according to silhouette score

In [ ]:
max_clusters = silhouette_k
cluster_labels = fcluster(row_linkage, max_clusters, criterion='maxclust')

# Create a DataFrame mapping each protein to its cluster.
clusters_df = pd.DataFrame({
    'Protein': pearson_clean.index,
    'Cluster': cluster_labels
})

print("\nProtein Clusters:")
print(clusters_df)

# Save the cluster assignments to a CSV file.
clusters_df.to_csv("results/clusters_pg/protein_clusters_sil.csv", index=False)
print("\nCluster assignments saved to 'results/clusters_pg/protein_clusters_sil.csv'.")

### Metrics on clusters

Compute Metrics different number of clusters to see the best, uncomment for analysis on 2-3-4 clust

In [ ]:
# # takes ~ 110 min to run


#  # Define range of cluster sizes
# k_range = range(2, 5)

# # Store results
# results = []
# wcss_values = []  # For the elbow method

# for k in k_range:
#     # Assign each protein to a cluster
#     clusters = sch.fcluster(row_linkage, k, criterion="maxclust")

#     # Convert to DataFrame
#     cluster_df = pd.DataFrame({"Protein": pearson_clean.index, "Cluster": clusters})

#     # Count proteins per cluster
#     cluster_sizes = cluster_df["Cluster"].value_counts().sort_index()

#     # Compute Within-Cluster Variance (WCSS)
#     wcss = sum(np.var(pearson_clean[clusters == i], axis=0).sum() for i in np.unique(clusters))
#     wcss_values.append(wcss)  # Store WCSS for elbow method

#     # Compute Silhouette Score
#     sil_score = silhouette_score(pearson_clean, clusters, metric='euclidean')

#     # Compute Dunn Index
#     dunn = corr_utils.dunn_index(pearson_clean.values, clusters)

#     # Compute Calinski-Harabasz Index
#     calinski_harabasz = calinski_harabasz_score(pearson_clean, clusters)

#     # Compute Davies-Bouldin Index
#     davies_bouldin = davies_bouldin_score(pearson_clean, clusters)

#     # Store results
#     results.append((k, cluster_sizes.to_dict(), wcss, sil_score, dunn, calinski_harabasz, davies_bouldin))

# # Print results
# for k, sizes, wcss, sil_score, dunn, calinski_harabasz, davies_bouldin in results:
#     print(f"\nClusters: {k}")
#     print(f"Cluster Sizes: {sizes}")
#     print(f"Within-Cluster Variance (WCSS): {wcss:.4f}")
#     print(f"Silhouette Score: {sil_score:.4f}")
#     print(f"Dunn Index: {dunn:.4f}")
#     print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")
#     print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")


Deeper analysis in 2-clusters

In [ ]:
clusters = sch.fcluster(row_linkage, 2, criterion="maxclust")
unique_clusters = np.unique(clusters)

cluster_stats = []

for cl in unique_clusters:
    # Get indices of proteins in the cluster
    cluster_indices = np.where(clusters == cl)[0]
    proteins = pearson_clean.index[cluster_indices]
    cluster_data = pearson_clean.iloc[cluster_indices, :]
    
    # Basic statistics
    size = len(proteins)
    centroid = cluster_data.mean(axis=0)
    intra_corr = cluster_data.values.flatten()  # all correlations within the cluster (can be filtered further)
    within_variance = cluster_data.var().sum()
    
    # Optionally, compute network-based or centrality measures here if needed
    
    cluster_stats.append({
        "Cluster": cl,
        "Size": size,
        "Centroid_Mean": centroid.mean(),  # overall average for centroid profile
        "Within_Variance": within_variance,
        "Intra_Cluster_Correlation_Mean": np.nanmean(intra_corr)
    })

# Convert to DataFrame for easier viewing
cluster_stats_df = pd.DataFrame(cluster_stats)
print(cluster_stats_df)

# Global clustering metrics (computed on all data with provided labels)
sil_score = silhouette_score(pearson_clean, clusters, metric='euclidean')
calinski_harabasz = calinski_harabasz_score(pearson_clean, clusters)
davies_bouldin = davies_bouldin_score(pearson_clean, clusters)

print(f"Global Silhouette Score: {sil_score:.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")



Deeper analysis in 3-Cluster

In [ ]:
clusters = sch.fcluster(row_linkage, 3, criterion="maxclust")
unique_clusters = np.unique(clusters)

cluster_stats = []

for cl in unique_clusters:
    # Get indices of proteins in the cluster
    cluster_indices = np.where(clusters == cl)[0]
    proteins = pearson_clean.index[cluster_indices]
    cluster_data = pearson_clean.iloc[cluster_indices, :]
    
    # Basic statistics
    size = len(proteins)
    centroid = cluster_data.mean(axis=0)
    intra_corr = cluster_data.values.flatten()  # all correlations within the cluster (can be filtered further)
    within_variance = cluster_data.var().sum()
    
    # Optionally, compute network-based or centrality measures here if needed
    
    cluster_stats.append({
        "Cluster": cl,
        "Size": size,
        "Centroid_Mean": centroid.mean(),  # overall average for centroid profile
        "Within_Variance": within_variance,
        "Intra_Cluster_Correlation_Mean": np.nanmean(intra_corr)
    })

# Convert to DataFrame for easier viewing
cluster_stats_df = pd.DataFrame(cluster_stats)
print(cluster_stats_df)

# Global clustering metrics (computed on all data with provided labels)
sil_score = silhouette_score(pearson_clean, clusters, metric='euclidean')
calinski_harabasz = calinski_harabasz_score(pearson_clean, clusters)
davies_bouldin = davies_bouldin_score(pearson_clean, clusters)

print(f"Global Silhouette Score: {sil_score:.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")



### PCA

PCA 2-clusters

In [ ]:
# PCA for 2-3 clusters
pca = PCA(n_components=2)
pca_result = pca.fit_transform(pearson_clean.values)
base_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=pearson_clean.index)

# Two clusterings
clusters2 = sch.fcluster(row_linkage, 2, criterion="maxclust")
clusters3 = sch.fcluster(row_linkage, 3, criterion="maxclust")
df2 = base_df.assign(Cluster=clusters2.astype(str))
df3 = base_df.assign(Cluster=clusters3.astype(str))

# Side-by-side plots
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

sns.scatterplot(data=df2, x="PC1", y="PC2", hue="Cluster", palette="Set1", s=20, ax=axes[0])
axes[0].set_title("2D PCA of Protein Clusters (k=2)")
axes[0].set_xlabel("Principal Component 1")
axes[0].set_ylabel("Principal Component 2")
axes[0].legend(title="Cluster", loc="best")

sns.scatterplot(data=df3, x="PC1", y="PC2", hue="Cluster", palette="Set1", s=20, ax=axes[1])
axes[1].set_title("2D PCA of Protein Clusters (k=3)")
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("")
axes[1].legend(title="Cluster", loc="best")

plt.tight_layout()
# Optional: save
# fig.savefig("results/pca_clusters_k2_k3.png", dpi=300, bbox_inches="tight")
plt.show()


PCA All Clusters

In [ ]:
# Create a 2x5 grid for plots with a reduced overall figure size
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()  # flatten to easily iterate over

# Loop over cluster numbers from 4 to 8
for i, k in enumerate(range(4, 8)):
    # Compute clusters using hierarchical clustering
    clusters = sch.fcluster(row_linkage, k, criterion="maxclust")
    
    # Perform PCA on the cleaned data
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(pearson_clean.values)
    
    # Create DataFrame for PCA results with cluster assignments
    pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=pearson_clean.index)
    pca_df["Cluster"] = clusters
    
    # Plot on the corresponding axis
    ax = axes[i]
    sns.scatterplot(
        data=pca_df, 
        x="PC1", 
        y="PC2", 
        hue="Cluster", 
        palette="Set1", 
        s=10, 
        ax=ax,
        legend=False  # Disable legend for individual subplots
    )
    ax.set_title(f"k = {k}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

#### Statistical analysis of clusters

In [ ]:

# Set a threshold for average gene correlation (example threshold: 0.5)
threshold = 0.1

# Create a copy of your data and add the cluster labels
data = pearson_clean.copy()
data["Cluster"] = clusters  # clusters should align with the rows of pearson_clean

# Dictionary to store summary statistics for each cluster
cluster_stats = {}

# Loop over each cluster
for cluster_label, group in data.groupby("Cluster"):
    # Remove the cluster column so that only gene correlation values remain
    group_data = group.drop(columns=["Cluster"])
    
    # Compute gene-wise statistics for the proteins in this cluster
    gene_means = group_data.mean(axis=0)
    gene_vars = group_data.var(axis=0)
    gene_min = group_data.min(axis=0)
    gene_max = group_data.max(axis=0)
    
    # Aggregate these gene-level stats to get cluster-level summary measures.
    # For example, we compute the average of the gene means, standard deviation, etc.
    cluster_summary = {
        "mean_of_gene_means": gene_means.mean(),
        "std_of_gene_means": gene_means.std(),
        "mean_of_gene_vars": gene_vars.mean(),
        "std_of_gene_vars": gene_vars.std(),
        "average_min": gene_min.mean(),
        "average_max": gene_max.mean(),
        "num_genes_above_threshold": (gene_means > threshold).sum(),
        "total_genes": len(gene_means)
    }
    
    cluster_stats[cluster_label] = cluster_summary

# Convert the dictionary to a DataFrame for easier viewing
stats_df = pd.DataFrame(cluster_stats).T
stats_df.index.name = "Cluster"
print(stats_df)


2 clusters

In [ ]:
clusters = sch.fcluster(row_linkage, 2, criterion="maxclust")
threshold = 0.1

# Create a copy of your data and add the cluster labels
data = pearson_clean.copy()
data["Cluster"] = clusters  # clusters should align with the rows of pearson_clean

# Dictionary to store summary statistics for each cluster
cluster_stats = {}

# Loop over each cluster
for cluster_label, group in data.groupby("Cluster"):
    # Remove the cluster column so that only gene correlation values remain
    group_data = group.drop(columns=["Cluster"])
    
    # Compute gene-wise statistics for the proteins in this cluster
    gene_means = group_data.mean(axis=0)
    gene_vars = group_data.var(axis=0)
    gene_min = group_data.min(axis=0)
    gene_max = group_data.max(axis=0)
    
    # Aggregate these gene-level stats to get cluster-level summary measures.
    # For example, we compute the average of the gene means, standard deviation, etc.
    cluster_summary = {
        "mean_of_gene_means": gene_means.mean(),
        "std_of_gene_means": gene_means.std(),
        "mean_of_gene_vars": gene_vars.mean(),
        "std_of_gene_vars": gene_vars.std(),
        "average_min": gene_min.mean(),
        "average_max": gene_max.mean(),
        "num_genes_above_threshold": (gene_means > threshold).sum(),
        "total_genes": len(gene_means)
    }
    
    cluster_stats[cluster_label] = cluster_summary

# Convert the dictionary to a DataFrame for easier viewing
stats_df = pd.DataFrame(cluster_stats).T
stats_df.index.name = "Cluster"
print(stats_df)


### Other analyses on 2, 3 clusters

#### 2 clusters

Violin Plots & distributions

In [ ]:
# Clusters (k=2)
clusters = sch.fcluster(row_linkage, 2, criterion="maxclust")

# ----- Prep data once -----
data = pearson_clean.copy()
data.index.name = "Protein"
data["Cluster"] = clusters

# Long form for violin plot
data_long = (
    data.reset_index()
        .melt(id_vars=["Protein", "Cluster"], var_name="Gene", value_name="Correlation")
)
data_long["Cluster"] = data_long["Cluster"].astype(str)

# Per-cluster gene averages for histogram
cluster_averages = data.groupby("Cluster")[pearson_clean.columns].mean()
avg_long = (
    cluster_averages.reset_index()
        .melt(id_vars="Cluster", var_name="Gene", value_name="Avg_Correlation")
)
avg_long["Cluster"] = avg_long["Cluster"].astype(str)

# ----- Plots side-by-side -----
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Violin
sns.violinplot(data=data_long, x="Cluster", y="Correlation", palette="Set1", cut=0, ax=axes[0])
axes[0].set_title("Protein–Gene Correlations per Cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Correlation")
axes[0].set_ylim(-0.3, 0.3)

# (b) Histogram / KDE of per-cluster gene averages
sns.histplot(data=avg_long, x="Avg_Correlation", hue="Cluster",
             palette="Set1", kde=True, element="step", ax=axes[1])
axes[1].set_title("Average Gene Correlations per Cluster")
axes[1].set_xlabel("Average Correlation")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


#### 3-clusters

In [ ]:
# Cluster labels (k=3)
clusters = sch.fcluster(row_linkage, 3, criterion="maxclust")

# Prep data
data = pearson_clean.copy()
data.index.name = "Protein"
data["Cluster"] = clusters.astype(str)

# Long form for violin
data_long = (
    data.reset_index()
        .melt(id_vars=["Protein", "Cluster"], var_name="Gene", value_name="Correlation")
)

# Per-cluster gene averages for histogram
cluster_averages = data.groupby("Cluster")[pearson_clean.columns].mean()
avg_long = (
    cluster_averages.reset_index()
        .melt(id_vars="Cluster", var_name="Gene", value_name="Avg_Correlation")
)

# --- Plots side-by-side ---
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Violin
sns.violinplot(data=data_long, x="Cluster", y="Correlation", palette="Set1", cut=0, ax=axes[0])
axes[0].set_title("Violin: Protein–Gene Correlations per Cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Correlation")
axes[0].set_ylim(-0.3, 0.3)

# (b) Histogram / KDE of per-cluster gene averages
sns.histplot(data=avg_long, x="Avg_Correlation", hue="Cluster",
             palette="Set1", kde=True, element="step", ax=axes[1])
axes[1].set_title("Histogram/KDE: Avg Gene Correlations per Cluster")
axes[1].set_xlabel("Average Correlation")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

 ## Correlation networks ## This can be dropped

### Protein - Gene

In [ ]:
#pip install community
#pip install python-louvain

#### 2 - clusters

In [ ]:
clusters = sch.fcluster(row_linkage, 2, criterion="maxclust")

In [ ]:
# ----- 3. Correlation Network Analysis ----- #

# Build a network where nodes are proteins and an edge exists if the similarity between their gene correlation profiles is high.
# Here, compute the protein–protein correlation matrix using the rows of pearson_clean.
protein_corr = pearson_clean.T.corr()  # proteins x proteins

# Set a correlation threshold for connecting nodes 
corr_threshold = 0.7

# Create an undirected graph
G = nx.Graph()

# Map each protein (row) to its cluster label
protein_cluster_mapping = dict(zip(pearson_clean.index, clusters))

# Add nodes with cluster attribute
for protein in protein_corr.index:
    G.add_node(protein, cluster=protein_cluster_mapping[protein])

# Add edges between proteins with correlation above the threshold
for i, protein_i in enumerate(protein_corr.index):
    for j, protein_j in enumerate(protein_corr.columns):
        if i < j and protein_corr.loc[protein_i, protein_j] > corr_threshold:
            G.add_edge(protein_i, protein_j, weight=protein_corr.loc[protein_i, protein_j])

# Compute network density
density = nx.density(G)
print("\nNetwork Density:", density)

# For modularity, we can use the community_louvain package (install via pip install python-louvain) 
partition = community_louvain.best_partition(G)
modularity = community_louvain.modularity(partition, G)
print("Network Modularity:", modularity)


# Identify hub proteins (top 5 by degree centrality)
degree_centrality = nx.degree_centrality(G)
sorted_hubs = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Hub Proteins (by Degree Centrality):")
for protein, centrality in sorted_hubs[:5]:
    print(f"{protein}: {centrality:.4f}")

# Optionally, visualize the network (this can be messy for large networks)
plt.figure(figsize=(10,8))
pos = nx.spring_layout(G, seed=42)  # positions for all nodes
node_colors = [G.nodes[node]['cluster'] for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=50, cmap=plt.cm.Set1, node_color=node_colors)
nx.draw_networkx_edges(G, pos, alpha=0.3)
plt.title("Protein Correlation Network (Edges with r > {:.2f})".format(corr_threshold))
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Create zoomed version of central cluster
plt.figure(figsize=(12, 10))

# Get the same layout as the original visualization for consistency
pos = nx.spring_layout(G, seed=42)  # Use same seed as original for consistency

# Get the approximate center coordinates
x_coords = [coord[0] for coord in pos.values()]
y_coords = [coord[1] for coord in pos.values()]
center_x = sum(x_coords) / len(x_coords)
center_y = sum(y_coords) / len(y_coords)

# Define radius for central region
radius = 0.3  # Adjust this value to control zoom level

# Identify nodes in the central region
central_nodes = [node for node in G.nodes() if 
                ((pos[node][0] - center_x)**2 + (pos[node][1] - center_y)**2) < radius**2]

# Draw edges with increased transparency for better visibility
central_edge_list = [(u, v) for u, v in G.edges() if u in central_nodes and v in central_nodes]
central_G = nx.Graph()
central_G.add_edges_from(central_edge_list)
nx.draw_networkx_edges(central_G, pos, alpha=0.2, width=0.5)

# Draw high correlation edges more prominently
high_corr_edges = [(u, v) for u, v, d in G.edges(data=True) 
                  if d['weight'] > 0.8 and u in central_nodes and v in central_nodes]
high_corr_G = nx.Graph()
high_corr_G.add_edges_from(high_corr_edges)
nx.draw_networkx_edges(high_corr_G, pos, alpha=0.6, width=1.0, edge_color='blue')

# Draw nodes for central region
central_nodes_colors = [G.nodes[node]['cluster'] for node in central_nodes]
nx.draw_networkx_nodes(G, pos, nodelist=central_nodes, node_size=15, 
                      alpha=0.8, cmap=plt.cm.Set1, node_color=central_nodes_colors)

# Add labels for hub proteins in central region (top 10)
top_hubs = [protein for protein, _ in sorted_hubs[:10]]
#central_hub_labels = {node: node for node in top_hubs if node in central_nodes}
#nx.draw_networkx_labels(G, pos, labels=central_hub_labels, font_size=10, font_color='black', font_weight='bold')

# Set plot limits for zoom effect
plt.xlim(center_x - radius, center_x + radius)
plt.ylim(center_y - radius, center_y + radius)

plt.title("Zoomed View of Central Cluster (High-Correlation Proteins)")
plt.axis("off")
plt.tight_layout()
plt.savefig("results/clusters_pg/zoomed_central_cluster.png", dpi=300)
plt.show()

In [ ]:
# 1. Save the network assignments for each protein
cluster_df = pd.DataFrame({
    'Protein': list(protein_cluster_mapping.keys()),
    'Cluster': list(protein_cluster_mapping.values())
})

# 2. Save community detection results (Louvain algorithm)
if 'partition' in locals():
    community_df = pd.DataFrame({
        'Protein': list(partition.keys()),
        'Community': list(partition.values())
    })
    community_df.to_csv('results/clusters_pg/protein_communities.csv', index=False)

# 3. Save network properties of each protein
# Degree centrality
degree_df = pd.DataFrame({
    'Protein': list(degree_centrality.keys()),
    'DegreeCentrality': list(degree_centrality.values())
})

# You could also calculate other centrality measures
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)

# Add these measures to the dataframe
degree_df['BetweennessCentrality'] = [betweenness_centrality.get(p, 0) for p in degree_df['Protein']]
degree_df['ClosenessCentrality'] = [closeness_centrality.get(p, 0) for p in degree_df['Protein']]

# Add cluster information
degree_df['Cluster'] = [protein_cluster_mapping.get(p, -1) for p in degree_df['Protein']]

# Save to CSV
degree_df.to_csv('results/clusters_pg/protein_network_properties.csv', index=False)

# 4. Save the network edges to a file
edges_list = list(G.edges(data=True))
edge_df = pd.DataFrame([
    {'Protein1': p1, 'Protein2': p2, 'Weight': data.get('weight', 0)} 
    for p1, p2, data in edges_list
])
edge_df.to_csv('results/clusters_pg/protein_network_edges.csv', index=False)

# 5. For more advanced analysis, you could save as GraphML format
# which preserves the full network structure for loading into tools like Cytoscape
nx.write_graphml(G, 'results/clusters_pg/protein_correlation_network.graphml')
print("All network analysis results saved to CSV files")

#### 3 - clusters

In [ ]:
clusters = sch.fcluster(row_linkage, 3, criterion="maxclust")

In [ ]:
# ----- 3. Correlation Network Analysis ----- #

# Build a network where nodes are proteins and an edge exists if the similarity between their gene correlation profiles is high.
# Here, compute the protein–protein correlation matrix using the rows of pearson_clean.
protein_corr = pearson_clean.T.corr()  # proteins x proteins

# Set a correlation threshold for connecting nodes 
corr_threshold = 0.7

# Create an undirected graph
G = nx.Graph()

# Map each protein (row) to its cluster label
protein_cluster_mapping = dict(zip(pearson_clean.index, clusters))

# Add nodes with cluster attribute
for protein in protein_corr.index:
    G.add_node(protein, cluster=protein_cluster_mapping[protein])

# Add edges between proteins with correlation above the threshold
for i, protein_i in enumerate(protein_corr.index):
    for j, protein_j in enumerate(protein_corr.columns):
        if i < j and protein_corr.loc[protein_i, protein_j] > corr_threshold:
            G.add_edge(protein_i, protein_j, weight=protein_corr.loc[protein_i, protein_j])

# Compute network density
density = nx.density(G)
print("\nNetwork Density:", density)

# For modularity, we can use the community_louvain package (install via pip install python-louvain) 
partition = community_louvain.best_partition(G)
modularity = community_louvain.modularity(partition, G)
print("Network Modularity:", modularity)

# Identify hub proteins (top 5 by degree centrality)
degree_centrality = nx.degree_centrality(G)
sorted_hubs = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Hub Proteins (by Degree Centrality):")
for protein, centrality in sorted_hubs[:5]:
    print(f"{protein}: {centrality:.4f}")

# Optionally, visualize the network (this can be messy for large networks)
plt.figure(figsize=(10,8))
pos = nx.spring_layout(G, seed=42)  # positions for all nodes
node_colors = [G.nodes[node]['cluster'] for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=50, cmap=plt.cm.Set1, node_color=node_colors)
nx.draw_networkx_edges(G, pos, alpha=0.3)
plt.title("Protein Correlation Network (Edges with r > {:.2f})".format(corr_threshold))
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Create zoomed version of central cluster
plt.figure(figsize=(12, 10))

# Get the same layout as the original visualization for consistency
pos = nx.spring_layout(G, seed=42)  # Use same seed as original for consistency

# Get the approximate center coordinates
x_coords = [coord[0] for coord in pos.values()]
y_coords = [coord[1] for coord in pos.values()]
center_x = sum(x_coords) / len(x_coords)
center_y = sum(y_coords) / len(y_coords)

# Define radius for central region
radius = 0.3  # Adjust this value to control zoom level

# Identify nodes in the central region
central_nodes = [node for node in G.nodes() if 
                ((pos[node][0] - center_x)**2 + (pos[node][1] - center_y)**2) < radius**2]

# Draw edges with increased transparency for better visibility
central_edge_list = [(u, v) for u, v in G.edges() if u in central_nodes and v in central_nodes]
central_G = nx.Graph()
central_G.add_edges_from(central_edge_list)
nx.draw_networkx_edges(central_G, pos, alpha=0.2, width=0.5)

# Draw high correlation edges more prominently
high_corr_edges = [(u, v) for u, v, d in G.edges(data=True) 
                  if d['weight'] > 0.9 and u in central_nodes and v in central_nodes]
high_corr_G = nx.Graph()
high_corr_G.add_edges_from(high_corr_edges)
nx.draw_networkx_edges(high_corr_G, pos, alpha=0.6, width=1.0, edge_color='lightyellow')


# Draw nodes for central region
central_nodes_colors = [G.nodes[node]['cluster'] for node in central_nodes]
nx.draw_networkx_nodes(G, pos, nodelist=central_nodes, node_size=15, 
                      alpha=0.8, cmap=plt.cm.Set1, node_color=central_nodes_colors)

# Add labels for hub proteins in central region (top 10)
top_hubs = [protein for protein, _ in sorted_hubs[:10]]
#central_hub_labels = {node: node for node in top_hubs if node in central_nodes}
#nx.draw_networkx_labels(G, pos, labels=central_hub_labels, font_size=10, font_color='black', font_weight='bold')

# Set plot limits for zoom effect
plt.xlim(center_x - radius, center_x + radius)
plt.ylim(center_y - radius, center_y + radius)

plt.title("Zoomed View of Central Cluster (High-Correlation Proteins)")
plt.axis("off")
plt.tight_layout()
plt.savefig("zoomed_central_cluster.png", dpi=300)
plt.show()